# Feature and gene-profile correlation analysis

S4 contains correlations across cleaned wells. S5 contains correlations across selected normalized gene profiles, signed-TOM modules, UMAP, and previously identified stationary-phase genes. Normalization and network calculations are shown below; the supplied full correlation matrix and embedding make default figure reproduction practical.

In [ ]:
from pathlib import Path
import numpy as np
import numpy.typing
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PACKAGE = Path.cwd()
if not (PACKAGE / "Datasets").is_dir():
    PACKAGE = PACKAGE.parent
DATASETS = PACKAGE / "Datasets"
if not DATASETS.is_dir():
    raise FileNotFoundError("Launch this notebook from the package root or Github folder.")
plt.rcParams.update({"font.family": "DejaVu Sans", "text.usetex": False})

import seaborn as sns
from pycytominer import aggregate, normalize, feature_select
from scipy.stats import spearmanr, gaussian_kde
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import squareform

book = DATASETS / "Dataset S5.xlsx"
rescaled_df = pd.read_excel(DATASETS / "Dataset S3.xlsx", sheet_name="Raw_well_profiles")
features = [c for c in rescaled_df if not c.startswith("Metadata_")]
agg_df = aggregate(population_df=rescaled_df, strata=["Metadata_plate","Metadata_gene"], features=features, operation="mean")
agg_ann_df = agg_df.copy()

## Aggregate, robustly normalize and select features
Average wells within plate/genotype, score against pooled WT profiles with mad_robustize, remove outliers/low-variance/highly correlated features, and average selected normalized profiles within deletion gene. WT is retained for normalization and UMAP but excluded from gene correlations.

In [ ]:
from pycytominer import normalize, feature_select
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
from sklearn.impute import KNNImputer
if 'Unnamed: 0' in rescaled_df.columns:
    rescaled_df = rescaled_df.drop(columns=['Unnamed: 0'])
features = [col for col in rescaled_df.columns if not col.startswith('Metadata_')]
norm_df = normalize(profiles=agg_ann_df, features=features, image_features=False, meta_features=['Metadata_gene', 'Metadata_plate'], samples="Metadata_gene == 'WT'", method='mad_robustize', output_file=None)
features = [ncol for ncol in norm_df.columns if not ncol.startswith('Metadata_')]
selected_df = feature_select(profiles=norm_df, features=features, image_features=False, samples='all', operation=['drop_outliers', 'variance_threshold', 'correlation_threshold'], na_cutoff=0.5, corr_threshold=0.97, corr_method='pearson', freq_cut=0.01, unique_cut=0.1, outlier_cutoff=100, output_file=None)
norm_features = [col for col in selected_df.columns if not col.startswith('Metadata_')]
gene_df = selected_df[selected_df['Metadata_gene'] != 'WT'].copy()
gene_profiles = gene_df.groupby('Metadata_gene')[norm_features].mean()

In [ ]:
reference_features = pd.read_excel(book, sheet_name="Selected_features").feature.tolist()
assert norm_features == reference_features

## Gene correlations and clustered heatmap
Spearman correlations use pairwise omission across morphology features. The optional full recalculation is slower than plotting.

In [ ]:
corr_df = pd.read_excel(book, sheet_name="Gene_correlations").set_index("Metadata_gene")
assert list(corr_df.index) == list(gene_profiles.index)
RECOMPUTE_CORRELATIONS = False
if RECOMPUTE_CORRELATIONS:
    # Pairwise-complete ranks are recomputed across selected features.
    # This full calculation can take substantially longer than plotting.
    values, _ = spearmanr(gene_profiles.to_numpy(), axis=1, nan_policy="omit")
    np.testing.assert_allclose(values, corr_df.to_numpy(), atol=2e-12, rtol=2e-12, equal_nan=True)
    corr_df = pd.DataFrame(values, index=gene_profiles.index, columns=gene_profiles.index)

In [ ]:
dist_matrix = 1 - corr_df.values
np.fill_diagonal(dist_matrix, 0)
dist_matrix = (dist_matrix + dist_matrix.T) / 2
condensed_dist = squareform(dist_matrix, checks=False)
linkage_matrix = linkage(condensed_dist, method='average')
dendro = dendrogram(linkage_matrix, no_plot=True)
order = corr_df.index.get_indexer(pd.read_excel(book, sheet_name='Gene_heatmap_order').Metadata_gene)
corr_clustered = corr_df.iloc[order, order]
(fig, ax) = plt.subplots(figsize=(10, 10))
im = ax.imshow(corr_clustered.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='equal')
ax.set_xticks([])
ax.set_yticks([])
cbar = plt.colorbar(im, ax=ax, label='Spearman ρ', shrink=0.8)
plt.title(f'Gene-Gene Spearman Correlations - Hierarchically Clustered (n={len(corr_df)} genes)')
plt.tight_layout()
plt.show()
mask = ~np.eye(corr_df.shape[0], dtype=bool)
off_diag = corr_df.values[mask]

## Signed adjacency, topological overlap and static modules
Signed adjacency uses beta8; TOM distance uses average linkage, cut0.935, and singleton reassignment for modules smaller than3.

In [ ]:
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
network_type = 'signed'
beta = 8
cut_height = 0.935
min_module_size = 3
small_module_mode = 'singleton'
feature_cols = [c for c in gene_profiles.columns if not str(c).startswith('Metadata_')]
X = gene_profiles[feature_cols].copy()
gene_order = X.index.intersection(corr_df.index)
X = X.loc[gene_order]
corr = corr_df.loc[gene_order, gene_order].to_numpy(dtype=np.float32, copy=True)
corr = np.array(corr, dtype=np.float32, copy=True)
corr.setflags(write=True)
corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
corr = np.clip((corr + corr.T) / 2, -1, 1)
np.fill_diagonal(corr, 1.0)

def adjacency_from_corr(corr, beta, network_type):
    if network_type == 'signed':
        A = np.clip((1 + corr) / 2, 0, 1) ** beta
    elif network_type == 'unsigned':
        A = np.abs(corr) ** beta
    else:
        raise ValueError("network_type must be 'signed' or 'unsigned'")
    A = A.astype(np.float32, copy=False)
    np.fill_diagonal(A, 0.0)
    return A

def compute_tom(A):
    L = A @ A
    k = A.sum(axis=1)
    denom = np.minimum.outer(k, k) + 1 - A
    TOM = np.divide(L + A, denom, out=np.zeros_like(A), where=denom > 0)
    TOM = np.clip((TOM + TOM.T) / 2, 0, 1).astype(np.float32)
    np.fill_diagonal(TOM, 1.0)
    return TOM

def compact_labels(labels):
    labels = np.asarray(labels, dtype=int)
    nonzero = [x for x in sorted(np.unique(labels)) if x != 0]
    mapper = {old: new for (new, old) in enumerate(nonzero, start=1)}
    if 0 in labels:
        mapper[0] = 0
    return np.array([mapper[x] for x in labels], dtype=int)

def clean_small_modules(labels, min_size=3, mode='singleton'):
    labels = np.asarray(labels, dtype=int).copy()
    sizes = pd.Series(labels).value_counts()
    small_labels = sizes[sizes < min_size].index.to_numpy()
    if len(small_labels) == 0:
        return compact_labels(labels)
    small_mask = np.isin(labels, small_labels)
    if mode == 'singleton':
        next_label = labels.max() + 1
        for idx in np.where(small_mask)[0]:
            labels[idx] = next_label
            next_label += 1
    elif mode == 'drop':
        labels[small_mask] = 0
    else:
        raise ValueError("small_module_mode must be 'singleton' or 'drop'")
    return compact_labels(labels)

def module_hubs(labels, A, gene_order):
    rows = []
    for module in sorted(np.unique(labels)):
        idx = np.where(labels == module)[0]
        genes = gene_order[idx]
        if len(idx) >= 2:
            k_within = A[np.ix_(idx, idx)].sum(axis=1)
            hub_pos = int(np.argmax(k_within))
            hub_gene = str(genes[hub_pos])
            hub_k = float(k_within[hub_pos])
        else:
            hub_gene = str(genes[0])
            hub_k = np.nan
        rows.append({'cluster': int(module), 'n_genes': int(len(idx)), 'hub_gene': hub_gene, 'hub_kWithin': hub_k})
    return pd.DataFrame(rows).sort_values(['n_genes', 'hub_kWithin'], ascending=[False, False]).reset_index(drop=True)
A = adjacency_from_corr(corr, beta=beta, network_type=network_type)
TOM = compute_tom(A)
tom_dist = 1 - TOM
tom_dist = np.clip((tom_dist + tom_dist.T) / 2, 0, 1)
np.fill_diagonal(tom_dist, 0.0)
tom_condensed = squareform(tom_dist, checks=False)
tom_linkage = linkage(tom_condensed, method='average')
raw_labels = fcluster(tom_linkage, t=float(cut_height), criterion='distance')
wgcna_labels = clean_small_modules(raw_labels, min_size=min_module_size, mode=small_module_mode)
wgcna_modules = pd.DataFrame({'Metadata_gene': gene_order, 'cluster': wgcna_labels, 'cluster_raw_static_cut': raw_labels.astype(int)})
gene_clusters = wgcna_modules.copy()
wgcna_hubs = module_hubs(wgcna_labels, A, gene_order)
run_summary = pd.DataFrame([{'network_type': network_type, 'beta': beta, 'cut_height': cut_height, 'min_module_size': min_module_size, 'small_module_mode': small_module_mode, 'n_genes': len(gene_order), 'n_total_clusters': wgcna_modules['cluster'].nunique(), 'n_multi_gene_clusters': int((wgcna_hubs['n_genes'] > 1).sum())}])
reference_modules = pd.read_excel(book, sheet_name="Module_members")
pd.testing.assert_frame_equal(wgcna_modules, reference_modules, check_dtype=False)
display(run_summary)

## UMAP and morphology overlays
Default coordinates are supplied. Optional reconstruction uses 10 neighbors, min_dist0.1 and seed42. Each overlay joins by plate and gene so every profile uses its own feature value; the join is one-to-one at the plate/genotype level.

In [ ]:
umap_data = pd.read_excel(book, sheet_name="UMAP_plotted")
umap_df = umap_data[["UMAP1","UMAP2","Metadata_gene","Metadata_plate"]].copy()
RECOMPUTE_UMAP = False
if RECOMPUTE_UMAP:
    from umap import UMAP
    X = selected_df[norm_features].fillna(selected_df[norm_features].median())
    model = UMAP(n_components=2,n_neighbors=10,min_dist=0.1,random_state=42,n_jobs=1)
    umap_df[["UMAP1","UMAP2"]] = model.fit_transform(X)

# S5 uses the original full-embedding grid, normalized over that grid.
wt = umap_df.loc[umap_df.Metadata_gene.str.contains("WT",na=False),["UMAP1","UMAP2"]]
kde = gaussian_kde(wt.T)
x = np.linspace(umap_df.UMAP1.min(),umap_df.UMAP1.max(),300)
y = np.linspace(umap_df.UMAP2.min(),umap_df.UMAP2.max(),300)
Xg,Yg = np.meshgrid(x,y)
Z = kde(np.vstack([Xg.ravel(),Yg.ravel()])).reshape(Xg.shape)
area = (x[1]-x[0])*(y[1]-y[0])
Z = Z/(Z.sum()*area)
descending = np.sort(Z.ravel())[::-1]
mass = np.cumsum(descending)*area
levels = [descending[np.searchsorted(mass,p)] for p in (0.9,0.75,0.5,0.25,0.1)]
density = {"Xg":Xg,"Yg":Yg,"Z":Z,"levels":levels}

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def plot_feature_on_umap(umap_df, density=None, agg_ann_df=None, feature=None, clip_percentiles=(0.5, 99.5), point_size=7, cmap='cividis', fixed_color=None, rasterize_scatter=True):
    """
    If feature is None:
        plot plain UMAP and, if provided, WT contours.
    If feature is not None:
        plot UMAP colored by feature, without WT contours.
    """
    plot_df = umap_df.copy()
    if feature is not None:
        if agg_ann_df is None:
            raise ValueError('agg_ann_df is required when feature is provided')
        if feature not in agg_ann_df.columns:
            raise ValueError(f"Feature '{feature}' not found in agg_ann_df")
        plot_df = plot_df.merge(agg_ann_df[['Metadata_plate', 'Metadata_gene', feature]], on=['Metadata_plate', 'Metadata_gene'], how='left', validate='one_to_one')
    fig = plt.figure(figsize=(5.5, 5))
    ax = fig.add_axes([0.18, 0.2, 0.62, 0.62])
    cax = None
    if feature is not None and pd.api.types.is_numeric_dtype(plot_df[feature]):
        cax = fig.add_axes([0.33, 0.09, 0.32, 0.022])
    if feature is None and density is not None:
        ax.contour(density['Xg'], density['Yg'], density['Z'], levels=density['levels'], colors='black', linewidths=0.9, alpha=0.7, zorder=20)
    if feature is None:
        ax.scatter(plot_df['UMAP1'], plot_df['UMAP2'], color=fixed_color if fixed_color is not None else 'steelblue', s=point_size, alpha=0.8, linewidths=0, zorder=10, rasterized=rasterize_scatter)
        title = 'UMAP'
    else:
        vals = plot_df[feature]
        is_numeric = pd.api.types.is_numeric_dtype(vals)
        if is_numeric:
            vals_num = pd.to_numeric(vals, errors='coerce')
            if fixed_color is not None:
                ax.scatter(plot_df['UMAP1'], plot_df['UMAP2'], color=fixed_color, s=point_size, alpha=0.8, linewidths=0, zorder=10, rasterized=rasterize_scatter)
                if cax is not None:
                    cax.set_axis_off()
            else:
                if clip_percentiles is not None:
                    (vmin, vmax) = np.nanpercentile(vals_num, clip_percentiles)
                    vals_plot = vals_num.clip(vmin, vmax)
                else:
                    vals_plot = vals_num
                sc = ax.scatter(plot_df['UMAP1'], plot_df['UMAP2'], c=vals_plot, cmap=cmap, s=point_size, alpha=0.8, linewidths=0, zorder=10, rasterized=rasterize_scatter)
                cbar = fig.colorbar(sc, cax=cax, orientation='horizontal')
                cbar.set_label(feature)
                cbar.outline.set_visible(True)
                cbar.outline.set_linewidth(0.8)
                cbar.ax.tick_params(length=2, pad=2, labelsize=8)
        elif fixed_color is not None:
            ax.scatter(plot_df['UMAP1'], plot_df['UMAP2'], color=fixed_color, s=point_size, alpha=0.9, linewidths=0, zorder=10, rasterized=rasterize_scatter)
        else:
            categories = vals.astype(str).unique()
            palette = sns.color_palette('tab20', len(categories))
            for (cat, color) in zip(categories, palette):
                idx = vals.astype(str) == cat
                ax.scatter(plot_df.loc[idx, 'UMAP1'], plot_df.loc[idx, 'UMAP2'], label=cat, color=color, s=point_size, alpha=0.9, linewidths=0, zorder=5, rasterized=rasterize_scatter)
            ax.legend(title=feature, bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
        title = f'UMAP colored by {feature}'
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.set_title(title, pad=10)
    xpad = 0.1 * (plot_df['UMAP1'].max() - plot_df['UMAP1'].min())
    ypad = 0.1 * (plot_df['UMAP2'].max() - plot_df['UMAP2'].min())
    ax.set_xlim(plot_df['UMAP1'].min() - xpad, plot_df['UMAP1'].max() + xpad)
    ax.set_ylim(plot_df['UMAP2'].min() - ypad, plot_df['UMAP2'].max() + ypad)
    sns.despine(ax=ax, top=True, right=True, left=True, bottom=True)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.show()

In [ ]:
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
from pathlib import Path

def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=256):
    new_cmap = LinearSegmentedColormap.from_list(f'trunc({cmap.name},{minval:.2f},{maxval:.2f})', cmap(np.linspace(minval, maxval, n)))
    return new_cmap
cmaps = truncate_colormap(plt.get_cmap('YlGnBu'), 0.2, 1)
cmaps = 'viridis'
plot_feature_on_umap(umap_df, density, agg_ann_df, feature=None, fixed_color='goldenrod')
plot_feature_on_umap(umap_df, density, agg_ann_df, 'cell_width', cmap=cmaps)
plot_feature_on_umap(umap_df, density, agg_ann_df, 'cell_length', cmap=cmaps)
plot_feature_on_umap(umap_df, density, agg_ann_df, 'C5_NC_ratio', cmap=cmaps)
plot_feature_on_umap(umap_df, density, agg_ann_df, 'C4_cell_mesh_intensity_mean', cmap=cmaps)
plot_feature_on_umap(umap_df, density, agg_ann_df, 'C3_cell_mesh_intensity_mean', cmap=cmaps)
plot_feature_on_umap(umap_df, density, agg_ann_df, 'C2_cell_glcm_energy', cmap=cmaps)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
largest_n_modules = 20
min_module_size = 3
exclude_cluster_0 = True
point_size_background = 7
point_size_module = 7
figsize = (5.5, 5)
ax_position = [0.18, 0.2, 0.62, 0.62]
modules_df = wgcna_modules[['Metadata_gene', 'cluster']].copy()
modules_df['cluster'] = pd.to_numeric(modules_df['cluster'], errors='coerce')
if exclude_cluster_0:
    modules_for_count = modules_df[modules_df['cluster'].ne(0)].copy()
else:
    modules_for_count = modules_df.copy()
module_sizes = modules_for_count.dropna(subset=['cluster']).groupby('cluster', as_index=False).agg(n_genes=('Metadata_gene', 'nunique')).query('n_genes >= @min_module_size').sort_values('n_genes', ascending=False).head(largest_n_modules)
clusters_to_plot = module_sizes['cluster'].astype(int).tolist()
plot_df = umap_df.copy()
plot_df = plot_df.merge(modules_df, on='Metadata_gene', how='left')
plot_df['cluster'] = pd.to_numeric(plot_df['cluster'], errors='coerce')
plot_df['is_largest_module'] = plot_df['cluster'].isin(clusters_to_plot)
sns.set_theme(style='white', context='talk')
fig = plt.figure(figsize=figsize)
ax = fig.add_axes(ax_position)
if 'density' in globals() and density is not None:
    ax.contour(density['Xg'], density['Yg'], density['Z'], levels=density['levels'], colors='black', linewidths=0.9, alpha=0.5, zorder=20)
bg = plot_df[~plot_df['is_largest_module']]
ax.scatter(bg['UMAP1'], bg['UMAP2'], color='lightgrey', s=point_size_background, alpha=0.35, linewidths=0, zorder=1, rasterized=True)
palette = sns.color_palette('tab20', n_colors=max(1, len(clusters_to_plot)))
color_map = dict(zip(clusters_to_plot, palette))
for cluster_id in clusters_to_plot:
    sub = plot_df[plot_df['cluster'].eq(cluster_id)]
    n_genes = int(module_sizes.loc[module_sizes['cluster'].eq(cluster_id), 'n_genes'].iloc[0])
    ax.scatter(sub['UMAP1'], sub['UMAP2'], color=color_map[cluster_id], s=point_size_module, alpha=0.9, linewidths=0, label=f'M{cluster_id} (n={n_genes})', zorder=10, rasterized=True)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.set_title('Largest static TOM/WGCNA modules on UMAP', pad=10)
xpad = 0.1 * (plot_df['UMAP1'].max() - plot_df['UMAP1'].min())
ypad = 0.1 * (plot_df['UMAP2'].max() - plot_df['UMAP2'].min())
ax.set_xlim(plot_df['UMAP1'].min() - xpad, plot_df['UMAP1'].max() + xpad)
ax.set_ylim(plot_df['UMAP2'].min() - ypad, plot_df['UMAP2'].max() + ypad)
sns.despine(ax=ax, top=True, right=True, left=True, bottom=True)
ax.set_xticks([])
ax.set_yticks([])
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=6, title='Modules', title_fontsize=7, markerscale=1.2, handletextpad=0.2, borderaxespad=0)
plt.show()

## Within-module gene neighborhoods
Select up to30 neighbors in the focal gene's module by absolute correlation, then average-linkage cluster their pairwise correlation distances.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['text.usetex'] = False

def plot_clean_module_spearman_dendrogram(focal_gene, corr_df=corr_df, wgcna_modules=wgcna_modules, max_module_neighbors=30, min_abs_r=None, same_module_only=True, figsize=(60 / 25.4, 70 / 25.4), label_fontsize=5.5, line_color='#4A4A4A', line_width=0.45):
    if focal_gene not in corr_df.index or focal_gene not in corr_df.columns:
        raise ValueError(f'{focal_gene!r} not found in corr_df')
    module_map = wgcna_modules[['Metadata_gene', 'cluster']].drop_duplicates('Metadata_gene').set_index('Metadata_gene')['cluster']
    focal_module = module_map.get(focal_gene, np.nan)
    if same_module_only:
        if pd.isna(focal_module):
            raise ValueError(f'{focal_gene!r} has no module assignment')
        candidate_genes = module_map[module_map.eq(focal_module)].index
        candidate_genes = candidate_genes.intersection(corr_df.index)
        candidate_genes = candidate_genes[candidate_genes != focal_gene]
    else:
        candidate_genes = corr_df.index[corr_df.index != focal_gene]
    focal_corr = corr_df.loc[focal_gene, candidate_genes].dropna()
    neighbor_df = pd.DataFrame({'gene': focal_corr.index, 'spearman_r_to_focal': focal_corr.values})
    neighbor_df['abs_r_to_focal'] = neighbor_df['spearman_r_to_focal'].abs()
    if min_abs_r is not None:
        neighbor_df = neighbor_df[neighbor_df['abs_r_to_focal'].ge(min_abs_r)].copy()
    neighbor_df = neighbor_df.sort_values('abs_r_to_focal', ascending=False).head(max_module_neighbors).reset_index(drop=True)
    if neighbor_df.empty:
        raise ValueError('No candidate genes selected.')
    selected_genes = [focal_gene] + neighbor_df['gene'].tolist()
    sub_corr = corr_df.loc[selected_genes, selected_genes].to_numpy(dtype=float)
    sub_corr = np.nan_to_num(sub_corr, nan=0.0)
    sub_corr = np.clip((sub_corr + sub_corr.T) / 2, -1, 1)
    np.fill_diagonal(sub_corr, 1.0)
    sub_dist = 1 - sub_corr
    sub_dist = np.clip((sub_dist + sub_dist.T) / 2, 0, 2)
    np.fill_diagonal(sub_dist, 0.0)
    Z = linkage(squareform(sub_dist, checks=False), method='average')
    labels = [f'Δ{g}' for g in selected_genes]
    (fig, ax) = plt.subplots(figsize=figsize, facecolor='white')
    dendro = dendrogram(Z, labels=labels, orientation='left', ax=ax, leaf_font_size=label_fontsize, color_threshold=0, above_threshold_color=line_color, link_color_func=lambda _: line_color)
    for collection in ax.collections:
        collection.set_linewidth(line_width)
    ax.set_title(f'Δ{focal_gene} — module {focal_module} candidates', fontsize=6.5, pad=4)
    ax.set_xlabel('1 - Spearman rho', fontsize=6)
    ax.set_ylabel('')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)
    ax.tick_params(axis='y', length=0, labelsize=label_fontsize)
    ax.tick_params(axis='x', labelsize=5.5, length=2, width=0.4)
    for tick in ax.get_yticklabels():
        tick.set_fontfamily('Arial')
        tick.set_fontstyle('italic')
        if tick.get_text() == f'Δ{focal_gene}':
            tick.set_fontweight('bold')
    for tick in ax.get_xticklabels():
        tick.set_fontfamily('Arial')
    with plt.rc_context({"font.size": 10}):
        plt.tight_layout(pad=0.4)
    plt.show()
    ordered_genes = [label.replace('Δ', '', 1) for label in dendro['ivl']]
    return (neighbor_df, ordered_genes, Z)
(neighbor_df, ordered_genes, Z) = plot_clean_module_spearman_dendrogram(focal_gene='yoaC', max_module_neighbors=30, same_module_only=True, line_width=0.45, figsize=(60 / 25.4, 100 / 25.4))

In [ ]:
(neighbor_df, ordered_genes, Z) = plot_clean_module_spearman_dendrogram(focal_gene='rffH', max_module_neighbors=30, same_module_only=True, line_width=0.45)

In [ ]:
(neighbor_df, ordered_genes, Z) = plot_clean_module_spearman_dendrogram(focal_gene='pnp', max_module_neighbors=30, same_module_only=True, line_width=0.45, figsize=(60 / 25.4, 100 / 25.4))

## S4: selected feature correlations across wells

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
selected_features = ['cell_length', 'cell_width', 'cell_area', 'cell_volume', 'cell_surface_area', 'cell_length_robust_CV', 'cell_width_robust_CV', 'C5_cell_total_obj_area', 'C5_NC_ratio', 'C4_cell_total_obj_area', 'C4_NC_ratio', 'C3_NC_ratio_CV', 'C2_NC_ratio_CV']
feature_label_map = {'C4_cell_total_obj_area': 'RNA area', 'C4_NC_ratio': 'RNA-cytoplasmic ratio', 'C5_cell_total_obj_area': 'Nucleoid area', 'C5_NC_ratio': 'Nucleocytoplasmic ratio', 'C3_NC_ratio_CV': 'IM heterogeneity', 'C2_NC_ratio_CV': 'OM heterogeneity', 'cell_length': 'Cell length', 'cell_width': 'Cell width', 'cell_area': 'Cell area', 'fraction_cell_constriction_degree_width_axial_above_0.1': 'Fraction of constricting cells', 'fraction_C5_object_number_above_1': 'Fraction of cells with multiple nucleoids', 'cell_volume': 'Cell volume', 'cell_surface_area': 'Cell surface area', 'cell_length_robust_CV': 'Cell length rCV', 'cell_width_robust_CV': 'Cell width rCV'}
available_features = [f for f in selected_features if f in rescaled_df.columns]
missing_features = [f for f in selected_features if f not in rescaled_df.columns]
X = rescaled_df[available_features].apply(pd.to_numeric, errors='coerce')
(corr_matrix, p_matrix) = spearmanr(X, axis=0, nan_policy='omit')
plot_labels = [feature_label_map.get(feature, feature) for feature in available_features]
corr_df = pd.DataFrame(corr_matrix, index=plot_labels, columns=plot_labels)
pval_df = pd.DataFrame(p_matrix, index=plot_labels, columns=plot_labels)
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
import matplotlib as mpl
mpl.rcParams.update({'font.family': 'Arial', 'font.sans-serif': ['Arial'], 'axes.linewidth': 0.8, 'axes.labelsize': 10, 'axes.titlesize': 12, 'xtick.labelsize': 9, 'ytick.labelsize': 9})
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
(fig, ax) = plt.subplots(figsize=(7.8, 7.2), dpi=300)
heatmap = sns.heatmap(corr_df, mask=mask, cmap='RdBu_r', vmin=-1, vmax=1, center=0, square=True, linewidths=0.6, linecolor='white', annot=True, fmt='.2f', annot_kws={'fontsize': 7.5, 'fontfamily': 'Arial', 'color': 'black'}, cbar_kws={'label': 'Spearman rho', 'shrink': 0.72, 'pad': 0.025, 'ticks': [-1, -0.5, 0, 0.5, 1]}, ax=ax)
for text in ax.texts:
    (x, y) = text.get_position()
    col = int(round(x - 0.5))
    row = int(round(y - 0.5))
    if row == col:
        text.set_color('white')
        text.set_fontweight('bold')
    else:
        text.set_color('black')
ax.set_title('Selected Feature Spearman Correlations', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('')
ax.set_ylabel('')
ax.tick_params(axis='both', length=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor', fontsize=9, fontfamily='Arial')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9, fontfamily='Arial')
for spine in ax.spines.values():
    spine.set_visible(False)
cbar = heatmap.collections[0].colorbar
cbar.ax.tick_params(labelsize=8, length=3, width=0.8)
cbar.set_label('Spearman rho', fontsize=9, fontfamily='Arial')
plt.tight_layout()
plt.show()
display(corr_df.round(3))

In [ ]:
reference = pd.read_excel(DATASETS / "Dataset S4.xlsx", sheet_name="Feature_correlations").set_index("feature")
np.testing.assert_allclose(corr_df,reference,atol=2e-12,rtol=2e-12,equal_nan=True)

## Previously identified stationary-phase genes (S5)
Use the same selected normalized profiles. Compute pairwise-complete gene correlations and optimal average-linkage ordering for the previously identified stationary-phase gene heatmap.

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform
prior_sp_genes = pd.read_excel(book, sheet_name='Prior_SP_genes')['Gene'].dropna().astype(str).str.strip()
norm_features = [col for col in selected_df.columns if not col.startswith('Metadata_')]
gene_names = selected_df['Metadata_gene'].astype('string').str.strip()
gene_df = selected_df.loc[gene_names.isin(prior_sp_genes) & gene_names.ne('WT')].copy()
gene_df['Metadata_gene'] = gene_names.loc[gene_df.index]
gene_profiles = gene_df.groupby('Metadata_gene')[norm_features].mean().replace([np.inf, -np.inf], np.nan)
missing_genes = sorted(set(prior_sp_genes) - set(gene_profiles.index))
if min(gene_profiles.shape) < 2:
    raise ValueError('At least two genes and two features are required.')

def spearman_matrix(values, axis):
    (corr, _) = spearmanr(values, axis=axis, nan_policy='omit')
    if np.ndim(corr) == 0:
        corr = np.array([[1.0, corr], [corr, 1.0]])
    return np.asarray(corr, dtype=float)
corr_matrix = spearman_matrix(gene_profiles.values, axis=1)
corr_df = pd.DataFrame(corr_matrix, index=gene_profiles.index, columns=gene_profiles.index)

def correlation_linkage(corr, labels):
    bad = ~np.isfinite(corr)
    np.fill_diagonal(bad, False)
    if bad.any():
        affected = [str(labels[i]) for i in np.flatnonzero(bad.any(axis=1))]
        raise ValueError('Undefined correlations involving: ' + ', '.join(affected) + '. Check constant profiles or insufficient overlapping values.')
    distance = np.clip(1.0 - corr, 0.0, 2.0)
    distance = (distance + distance.T) / 2.0
    np.fill_diagonal(distance, 0.0)
    return linkage(squareform(distance), method='average', optimal_ordering=True)
row_linkage = correlation_linkage(corr_matrix, gene_profiles.index)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties
from scipy.cluster.hierarchy import leaves_list
size_mm = 120
font_size = 7 * (size_mm / 160)
ordered_genes = pd.read_excel(book, sheet_name='Prior_SP_plot_order').Gene
ordered_corr = corr_df.loc[ordered_genes, ordered_genes]
with sns.axes_style('white'), plt.rc_context({'font.family': 'Arial', 'font.size': font_size, 'text.usetex': False}):
    fig = plt.figure(figsize=(size_mm / 25.4, size_mm / 25.4))
    ax = fig.add_axes([0.14, 0.18, 0.68, 0.68])
    ax_cbar = fig.add_axes([0.88, 0.285, 0.018, 0.47])
    sns.heatmap(ordered_corr, ax=ax, cmap='RdBu_r', vmin=-1, vmax=1, center=0, square=True, xticklabels=False, yticklabels=False, linewidths=0, rasterized=False, cbar_ax=ax_cbar, cbar_kws={'orientation': 'vertical', 'ticks': [-1, -0.5, 0, 0.5, 1], 'label': 'Spearman ρ'})
    ax.set_xlabel('')
    ax.set_ylabel('')
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    gene_font = FontProperties(family='Arial', style='italic', size=font_size)

    def add_gene_label(gene, xy, transform, rotation):
        gene = str(gene)
        width_px = renderer.get_text_width_height_descent(gene, gene_font, ismath=False)[0]
        width_pt = width_px * 72 / fig.dpi
        if rotation == 0:
            word_offset = (-2, 0)
            delta_offset = (-2 - width_pt - 0.5, 0)
        else:
            word_offset = (0, -2)
            delta_offset = (0, -2 - width_pt - 0.5)
        for (text, offset, style) in [(gene, word_offset, 'italic'), ('Δ', delta_offset, 'normal')]:
            ax.annotate(text, xy=xy, xycoords=transform, xytext=offset, textcoords='offset points', rotation=rotation, rotation_mode='anchor', ha='right', va='center', fontfamily='Arial', fontstyle=style, fontsize=font_size, annotation_clip=False)
    for (i, gene) in enumerate(ordered_genes):
        add_gene_label(gene, (0, i + 0.5), ax.get_yaxis_transform(), 0)
        add_gene_label(gene, (i + 0.5, 0), ax.get_xaxis_transform(), 90)
    ax_cbar.tick_params(labelsize=font_size, length=2, width=0.45)
    ax_cbar.yaxis.label.set_size(font_size + 0.5)
    colorbar = ax.collections[0].colorbar
    colorbar.outline.set_visible(True)
    colorbar.outline.set_edgecolor('black')
    colorbar.outline.set_linewidth(0.4)
    plt.show()